 # Medium BIT-S4-DD8 training and HPO



 The medium BIT architecture is implemented in `src/satchange/models/bit.py`. This notebook does not redefine it: `create_change_model()` selects `BIT` from the model factory and passes the parameters from `configs/bit.yaml`.



 A fresh **Restart Kernel → Run All** is safe. Dataset preparation, full training, HPO, and tiny-overfit diagnostics run only when their explicit flags are enabled.

In [10]:
# %%
import json
from pathlib import Path
from uuid import uuid4

import torch
from torch.utils.data import DataLoader, Subset

from satchange import BCEDiceLoss, fit_binary_change_model
from satchange.data import (
    RandomFlipRotate,
    create_dataset_dataloader,
    prepare_levir_cd,
)
from satchange.models import count_model_parameters, create_change_model
from satchange.optimization import run_parameter_search
from satchange.utils import load_config


In [11]:
# CONFIG, PROJECT_ROOT etc.


current_directory = Path.cwd().resolve()
project_root = (
    current_directory.parent
    if current_directory.name == "notebooks"
    else current_directory
)
if not (project_root / "pyproject.toml").is_file():
    raise FileNotFoundError(
        "Run this notebook from the project root or notebooks directory."
    )

config_path = project_root / "configs" / "bit_resized.yaml"
config = load_config(config_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(config["project"]["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(config["project"]["seed"])

model = create_change_model(
    model_name=config["model"]["name"],
    parameters=config["model"]["parameters"],
).to(device)

print("Config:", config_path)
print("Device:", device)
print("Created model:", type(model).__name__)
print("Architecture parameters:", config["model"]["parameters"])
print("Parameter counts:", count_model_parameters(model))


Config: configs/bit_resized.yaml
Device: cuda
Created model: BIT
Architecture parameters: {'in_channels': 3, 'out_channels': 1, 'embedding_dim': 32, 'token_length': 4, 'num_heads': 8, 'encoder_head_dim': 64, 'decoder_head_dim': 64, 'encoder_depth': 1, 'decoder_depth': 8, 'dropout': 0.0, 'pretrained_backbone': False}
Parameter counts: {'total_parameters': 3495973, 'trainable_parameters': 3495973}


In [12]:
image_size = config["data"]["image_size"]
before = torch.rand(1, 3, image_size, image_size, device=device)
after = torch.rand(1, 3, image_size, image_size, device=device)
model.eval()
with torch.no_grad():
    logits = model(before, after)

expected_shape = (1, 1, image_size, image_size)
assert tuple(logits.shape) == expected_shape
assert torch.isfinite(logits).all()
print("Forward output:", logits.shape)


Forward output: torch.Size([1, 1, 256, 256])


 ## Dataset location and optional preparation



 The training and HPO blocks use `dataset_root` below. Enable `RUN_DATA_SETUP` only when LEVIR-CD must be downloaded, extracted, or validated.

In [13]:
# DATA

RUN_DATA_SETUP = False

dataset_root = project_root / config["dataset"]["prepared_directory"]
if RUN_DATA_SETUP:
    dataset_root = prepare_levir_cd(
        config=config,
        project_root=project_root,
    )
    print("Dataset ready:", dataset_root)
else:
    print("Dataset preparation skipped. Expected location:", dataset_root)


Dataset preparation skipped. Expected location: data/LEVIR-CD


In [14]:
THRESHOLDS = [round(value / 100, 2) for value in range(5, 96, 5)]
loss_config = config["training"]["loss"]

print("Thresholds:", THRESHOLDS)
print("Loss parameters:", loss_config)


Thresholds: [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]
Loss parameters: {'bce_weight': 0.3, 'dice_weight': 0.7, 'pos_weight': 10}


 ## Full medium BIT training



 Set `RUN_FULL_TRAINING = True` to train one medium BIT configuration. Each new run creates a fresh model, loss, optimizer, and loaders. Set `RESUME_FROM` to that run's `last.pt` to continue it; `NUMBER_OF_EPOCHS` then means additional epochs.

In [15]:
# RUN_FULL_TRAINING = False
# RUN_NAME = "levir_bit_medium_baseline_v1"
# NUMBER_OF_EPOCHS = config["training"]["epochs"]
# TRAINING_PRECISION = "fp32"
# RESUME_FROM = None
# TRAINING_OUTPUT_ROOT = project_root / "artifacts" / "training_runs"

# training_history = []

# if RUN_FULL_TRAINING:
#     if not dataset_root.is_dir():
#         raise FileNotFoundError(
#             f"LEVIR-CD was not found at {dataset_root}. "
#             "Run the dataset-preparation cell first."
#         )

#     train_transform = RandomFlipRotate(
#         horizontal_flip_probability=0.5,
#         vertical_flip_probability=0.5,
#     )
#     _, train_loader = create_dataset_dataloader(
#         dataset_root=dataset_root,
#         split="train",
#         image_size=config["data"]["image_size"],
#         batch_size=config["data"]["batch_size"],
#         num_workers=config["data"]["num_workers"],
#         transform=train_transform,
#         shuffle=True,
#         pin_memory=torch.cuda.is_available(),
#         seed=config["project"]["seed"],
#     )
#     _, validation_loader = create_dataset_dataloader(
#         dataset_root=dataset_root,
#         split="val",
#         image_size=config["data"]["image_size"],
#         batch_size=config["data"]["batch_size"],
#         num_workers=config["data"]["num_workers"],
#         pin_memory=torch.cuda.is_available(),
#     )

#     training_model = create_change_model(
#         model_name=config["model"]["name"],
#         parameters=config["model"]["parameters"],
#     ).to(device)
#     training_loss = BCEDiceLoss(
#         bce_weight=loss_config["bce_weight"],
#         dice_weight=loss_config["dice_weight"],
#         pos_weight=loss_config["pos_weight"],
#     ).to(device)
#     training_optimizer = torch.optim.AdamW(
#         training_model.parameters(),
#         lr=config["training"]["learning_rate"],
#         weight_decay=config["training"]["weight_decay"],
#     )

#     training_history = fit_binary_change_model(
#         model=training_model,
#         train_loader=train_loader,
#         validation_loader=validation_loader,
#         loss_function=training_loss,
#         optimizer=training_optimizer,
#         device=device,
#         number_of_epochs=NUMBER_OF_EPOCHS,
#         thresholds=THRESHOLDS,
#         output_root=TRAINING_OUTPUT_ROOT,
#         model_name=config["model"]["name"],
#         run_name=RUN_NAME,
#         checkpoint_interval=5,
#         config=config,
#         resume_from=RESUME_FROM,
#         scheduler=None,
#         precision=TRAINING_PRECISION,
#     )

#     best_training_result = max(
#         training_history,
#         key=lambda result: result["validation_f1"],
#     )
#     print("Best training result:", best_training_result)
# else:
#     print("Full training skipped; set RUN_FULL_TRAINING=True to enable it.")


 ## Medium BIT hyperparameter search



 Top-level keys such as `weight_decay` tune training. Architecture keys use the `model__` prefix and are moved into a fresh nested `model_parameters` dictionary for each trial. For example: `model__embedding_dim`, `model__token_length`, or `model__decoder_head_dim`.

In [27]:
MEDIUM_RESIZED_FINAL_PARAMETERS = {
    "model_name": "bit",

    "model_parameters": {
        "in_channels": 3,
        "out_channels": 1,
        "embedding_dim": 32,
        "token_length": 4,
        "num_heads": 8,
        "encoder_head_dim": 64,
        "decoder_head_dim": 64,
        "encoder_depth": 1,
        "decoder_depth": 8,
        "dropout": 0.0,
        "pretrained_backbone": False,
    },

    "image_size": 256,
    "batch_size": 16,

    "learning_rate": 3e-4,
    "weight_decay": 1e-3,
    "scheduler": "none",

    "positive_weight": 25,
    "bce_weight": 0.5,
    "dice_weight": 0.5,

    "precision": "fp32",
    "epochs": 100,

    "seeds": [
        42,
        123,
        777,
    ],
}


FINAL_MODEL_DIR = (
    project_root
    / "artifacts"
    / "final_models"
    / "medium_resized"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


with (
    FINAL_MODEL_DIR
    / "final_parameters.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        MEDIUM_RESIZED_FINAL_PARAMETERS,
        file,
        indent=2,
    )


print(
    "Saved:",
    FINAL_MODEL_DIR,
)

Saved: artifacts/final_models/medium_resized


In [1]:
# ============================================================
# LIGHT RESIZED → MEDIUM RESIZED: VALIDATION / TEST / BENCHMARKS
# Выполнить целиком в одной ячейке. Обучение не запускается.
# ============================================================

import gc
import hashlib
import json
import shutil
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import torch

from satchange import BCEDiceLoss
from satchange.data import create_dataset_dataloader
from satchange.models import count_model_parameters, create_change_model
from satchange.training import evaluate_thresholds
from satchange.utils import load_config

METRICS = ("loss", "precision", "recall", "f1", "iou", "accuracy")
SEEDS = (42, 123, 777)


def _save_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False),
        encoding="utf-8",
    )


def _make_report(variant, model_name, split, rows, sample_count):
    report = {
        "model": model_name,
        "variant": variant,
        "split": split,
        "image_size": 256,
        "number_of_pairs": sample_count,
        "seeds": list(SEEDS),
        "evaluation_batch_size": 16,
        "num_workers": 0,
        "precision": "fp32",
        "threshold_selection": (
            "saved validation thresholds; no new threshold search"
        ),
        "std_type": "sample",
        "std_ddof": 1,
        "results": rows,
    }

    for metric in METRICS:
        key = f"{split}_{metric}"
        values = [row[key] for row in rows]
        report[f"{key}_mean"] = statistics.mean(values)
        report[f"{key}_std"] = (
            statistics.stdev(values) if len(values) > 1 else None
        )

    return report


@torch.inference_mode()
def _measure_forward(model, device, baseline, warmup=20, repetitions=100):
    # Входы уже на устройстве.
    # Чтение файлов, resize и thresholding не измеряются.
    model.eval()
    before = torch.zeros(1, 3, 256, 256, device=device)
    after = torch.zeros_like(before)

    for _ in range(warmup):
        output = model(before, after)
        del output

    if device.type == "cuda":
        torch.cuda.synchronize(device)
        torch.cuda.reset_peak_memory_stats(device)

    times = []

    for _ in range(repetitions):
        if device.type == "cuda":
            torch.cuda.synchronize(device)

        started = time.perf_counter()
        output = model(before, after)

        if device.type == "cuda":
            torch.cuda.synchronize(device)

        times.append((time.perf_counter() - started) * 1000)
        del output

    latency = statistics.mean(times)
    cuda = device.type == "cuda"

    return {
        "latency_ms_per_pair": latency,
        "latency_median_ms": statistics.median(times),
        "pairs_per_second": 1000 / latency,
        "latency_samples_ms": times,

        # Из пика выделенной памяти вычитаются уже существовавшие
        # GPU-тензоры notebook, но не память оцениваемой модели.
        "peak_inference_vram_mb": (
            (torch.cuda.max_memory_allocated(device) - baseline) / 2**20
            if cuda else None
        ),
        "notebook_baseline_vram_mb": (
            baseline / 2**20 if cuda else None
        ),
        "process_peak_allocated_vram_mb": (
            torch.cuda.max_memory_allocated(device) / 2**20
            if cuda else None
        ),
        "process_peak_reserved_vram_mb": (
            torch.cuda.max_memory_reserved(device) / 2**20
            if cuda else None
        ),

        "device": str(device),
        "gpu_name": (
            torch.cuda.get_device_name(device) if cuda else None
        ),
        "torch_version": str(torch.__version__),
        "cuda_version": torch.version.cuda,

        "protocol": {
            "precision": "fp32",
            "tf32": False,
            "batch_size": 1,
            "input_shape_per_image": [1, 3, 256, 256],
            "images_per_pair": 2,
            "warmup": warmup,
            "repetitions": repetitions,
            "timing": (
                "synchronized forward only; sequential batch=1 throughput"
            ),
            "memory_unit": (
                "MiB; existing notebook allocations subtracted "
                "from inference peak"
            ),
            "cudnn_benchmark": False,
            "cudnn_deterministic": True,
        },
    }


def _benchmark_one(
    root,
    variant,
    model_name,
    run_group,
    search_name,
    config_file,
    device,
):
    destination = root / "artifacts/final_models" / variant
    run_root = root / "artifacts" / run_group / search_name

    results_path = destination / "training_results.json"
    if not results_path.is_file():
        results_path = run_root / "results.json"

    source = json.loads(
        results_path.read_text(encoding="utf-8-sig")
    )

    runs = sorted(
        source["results"],
        key=lambda row: row["parameters"]["seed"],
    )

    if (
        source["model_name"] != model_name
        or source["search_name"] != search_name
        or [r["parameters"]["seed"] for r in runs] != list(SEEDS)
        or any(r["status"] != "completed" for r in runs)
    ):
        raise RuntimeError(
            f"Не совпали модель, финальный run или seeds: {results_path}"
        )

    architecture = runs[0]["parameters"]["model_parameters"]

    for row in runs:
        p = row["parameters"]

        if (
            p["image_size"] != 256
            or p["batch_size"] != 16
            or p.get("precision", "fp32") != "fp32"
            or p["model_parameters"] != architecture
        ):
            raise RuntimeError(
                f"Неожиданные параметры {variant}: {p}"
            )

    # Из YAML нужен только путь к данным.
    # Архитектура, loss и пороги берутся из финальных результатов.
    config = load_config(root / "configs" / config_file)
    dataset_root = root / config["dataset"]["prepared_directory"]

    work = (
        root / "artifacts/debug"
        / f"{variant}_benchmark_{uuid4().hex[:8]}"
    )
    staging = work / "ready"

    if destination.exists():
        shutil.copytree(destination, staging)
    else:
        staging.mkdir(parents=True)

    paths, hashes = {}, {}

    for row in runs:
        seed = row["parameters"]["seed"]
        path = destination / f"seed_{seed}_best.pt"

        if not path.is_file():
            path = (
                run_root
                / model_name
                / f"trial_{int(row['trial']):03d}"
                / "best.pt"
            )

        target = staging / f"seed_{seed}_best.pt"

        # Checkpoints из final_models уже скопированы в staging.
        if path.parent != destination:
            shutil.copy2(path, target)

        paths[str(seed)] = str(path.relative_to(root))

        with target.open("rb") as file:
            hashes[str(seed)] = hashlib.file_digest(
                file, "sha256"
            ).hexdigest()

    # Deployment выбирается ДО test, только по validation F1.
    winner = max(runs, key=lambda row: row["best_f1"])
    winner_seed = winner["parameters"]["seed"]

    print(f"\n{'=' * 60}")
    print(f"{variant} | {model_name} | 256×256 | FP32")
    print(
        f"Deployment seed: {winner_seed}; "
        f"threshold: {winner['best_threshold']}"
    )
    print("Рабочая папка:", work)

    gc.collect()

    if device.type == "cuda":
        with torch.cuda.device(device):
            torch.cuda.empty_cache()
        torch.cuda.synchronize(device)

    baseline = (
        torch.cuda.memory_allocated(device)
        if device.type == "cuda" else 0
    )

    model = create_change_model(
        model_name, architecture
    ).float().to(device)

    loss = None

    def restore(row):
        seed = row["parameters"]["seed"]

        # Только собственные доверенные checkpoints проекта.
        checkpoint = torch.load(
            staging / f"seed_{seed}_best.pt",
            map_location="cpu",
            weights_only=False,
        )

        metrics = checkpoint["current_metrics"]

        if (
            checkpoint["model_name"] != model_name
            or checkpoint["epoch"] != row["best_epoch"]
            or abs(
                metrics["best_threshold"] - row["best_threshold"]
            ) > 1e-12
            or any(
                abs(
                    metrics[f"validation_{m}"] - row[f"best_{m}"]
                ) > 1e-6
                for m in ("f1", "iou")
            )
        ):
            raise RuntimeError(
                f"Checkpoint не соответствует отчёту: "
                f"{variant}, seed={seed}"
            )

        saved = checkpoint.get("config", {})

        for key in (
            "seed",
            "image_size",
            "model_parameters",
            "batch_size",
            "positive_weight",
            "bce_weight",
            "dice_weight",
        ):
            if key in saved and saved[key] != row["parameters"][key]:
                raise RuntimeError(
                    f"Checkpoint: не совпал {key}, seed={seed}"
                )

        if checkpoint.get("precision", "fp32") != "fp32":
            raise RuntimeError(
                f"Checkpoint: неожиданная precision, seed={seed}"
            )

        model.load_state_dict(
            checkpoint["model_state_dict"],
            strict=True,
        )
        model.eval()

    reports = {}

    try:
        # Сначала validation ВСЕХ seeds.
        # Test начинается только после её успешного завершения.
        for split, filename in (
            ("validation", "validation_results.json"),
            ("test", "final_test_results.json"),
        ):
            dataset, loader = create_dataset_dataloader(
                dataset_root=dataset_root,
                split="val" if split == "validation" else "test",
                image_size=256,
                batch_size=16,
                num_workers=0,
                transform=None,
                shuffle=False,
                pin_memory=device.type == "cuda",
            )

            rows = []

            for run in runs:
                p = run["parameters"]
                restore(run)

                loss = BCEDiceLoss(
                    bce_weight=p["bce_weight"],
                    dice_weight=p["dice_weight"],
                    pos_weight=p["positive_weight"],
                ).to(device)

                result = evaluate_thresholds(
                    model=model,
                    data_loader=loader,
                    loss_function=loss,
                    device=device,
                    thresholds=[float(run["best_threshold"])],
                    precision="fp32",
                )

                metrics = {
                    "loss": result["loss"],
                    **result["threshold_results"][0],
                }

                row = {
                    "seed": p["seed"],
                    "trial": f"trial_{int(run['trial']):03d}",
                    "best_validation_epoch": run["best_epoch"],
                    "validation_threshold": run["best_threshold"],
                    "original_validation_f1": run["best_f1"],
                    "original_validation_iou": run["best_iou"],

                    **{
                        f"{split}_{key}": metrics[key]
                        for key in METRICS
                    },
                    **{
                        f"{split}_{key}": metrics[key]
                        for key in ("tp", "fp", "fn", "tn")
                    },
                }

                if split == "test":
                    row.update(
                        validation_f1=run["best_f1"],
                        validation_iou=run["best_iou"],
                    )

                rows.append(row)

                print(
                    f"{split:10s} seed={p['seed']:3d} | "
                    f"F1={metrics['f1']:.6f} | "
                    f"IoU={metrics['iou']:.6f}"
                )

                if split == "validation" and any(
                    abs(metrics[m] - run[f"best_{m}"]) > 1e-4
                    for m in ("f1", "iou")
                ):
                    _save_json(
                        work / "validation_mismatch.json",
                        rows,
                    )
                    raise RuntimeError(
                        f"Validation отличается от сохранённой: "
                        f"{variant}, seed={p['seed']}. "
                        f"Test не запущен. Смотри {work}"
                    )

            reports[split] = _make_report(
                variant,
                model_name,
                split,
                rows,
                len(dataset),
            )
            _save_json(staging / filename, reports[split])

        del loader, dataset
        loss = None

        # Веса deployment — только state_dict, без optimizer.
        restore(winner)

        torch.save(
            {
                key: value.detach().cpu()
                for key, value in model.state_dict().items()
            },
            staging / "deployment_weights.pt",
        )

        gc.collect()

        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()

        benchmark = _measure_forward(
            model, device, baseline
        )
        counts = count_model_parameters(model)

        benchmark.update(
            parameters=counts["total_parameters"],
            trainable_parameters=counts["trainable_parameters"],

            checkpoint_mb=(
                staging / f"seed_{winner_seed}_best.pt"
            ).stat().st_size / 2**20,

            weights_only_mb=(
                staging / "deployment_weights.pt"
            ).stat().st_size / 2**20,

            checkpoint_sizes_mib={
                str(seed): (
                    staging / f"seed_{seed}_best.pt"
                ).stat().st_size / 2**20
                for seed in SEEDS
            },
        )

        reports["benchmark"] = benchmark

        _save_json(staging / "benchmark.json", benchmark)
        _save_json(staging / "training_results.json", source)

        metadata_path = staging / "metadata.json"
        metadata = (
            json.loads(metadata_path.read_text())
            if metadata_path.is_file() else {}
        )

        metadata.update(
            model_level=variant.split("_")[0],
            variant="resized_256",
            model_name=model_name,
            image_size=256,
            model_parameters=architecture,

            deployment_seed=winner_seed,
            deployment_best_epoch=winner["best_epoch"],
            deployment_threshold=winner["best_threshold"],
            deployment_validation_f1=winner["best_f1"],
            deployment_validation_iou=winner["best_iou"],
            deployment_selection=(
                "highest saved validation F1; never test F1"
            ),

            source_results=str(results_path.relative_to(root)),
            checkpoint_sha256=hashes,
            dataset_root=str(dataset_root),
            evaluation_complete=True,
            evaluated_at_utc=datetime.now(timezone.utc).isoformat(),
            evaluation_function="satchange.training.evaluate_thresholds",

            seeds={
                str(r["parameters"]["seed"]): {
                    "trial": f"trial_{int(r['trial']):03d}",
                    "best_epoch": r["best_epoch"],
                    "validation_threshold": r["best_threshold"],
                    "validation_f1": r["best_f1"],
                    "validation_iou": r["best_iou"],
                    "validation_precision": r["best_precision"],
                    "validation_recall": r["best_recall"],
                    "parameters": r["parameters"],
                    "source_checkpoint": paths[
                        str(r["parameters"]["seed"])
                    ],
                }
                for r in runs
            },
        )

        _save_json(metadata_path, metadata)

        # Заменяем финальную папку только после завершения всех этапов.
        # Прежняя версия остаётся в artifacts/debug/.../previous.
        destination.parent.mkdir(parents=True, exist_ok=True)
        backup = work / "previous"

        if destination.exists():
            destination.rename(backup)

        try:
            staging.rename(destination)
        except BaseException:
            if backup.exists():
                backup.rename(destination)
            raise

        print("Сохранено:", destination)
        return reports

    finally:
        model = loss = None
        gc.collect()

        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()


def run_resized_benchmarks(project_root, device="cuda:0"):
    root = Path(project_root).resolve()
    device = torch.device(device)

    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA недоступна. Выбери GPU-kernel; "
            "CPU-оценка: device='cpu'."
        )

    old = (
        torch.backends.cudnn.benchmark,
        torch.backends.cudnn.deterministic,
        torch.backends.cuda.matmul.allow_tf32,
        torch.backends.cudnn.allow_tf32,
    )

    jobs = (
        (
            "light_resized",
            "fc_siam_diff",
            "training_runs",
            "fc_siam_diff_256_final_multiseed",
            "fc_siam_diff_resized.yaml",
        ),
        (
            "medium_resized",
            "bit",
            "hyperparameter_search",
            "bit_medium_resized_final_100ep_seeds42_123_777_v1",
            "bit_resized.yaml",
        ),
    )

    try:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False

        reports = {}

        # Сначала полностью Light, затем полностью Medium.
        for job in jobs:
            reports[job[0]] = _benchmark_one(
                root, *job, device
            )

        return reports

    finally:
        (
            torch.backends.cudnn.benchmark,
            torch.backends.cudnn.deterministic,
            torch.backends.cuda.matmul.allow_tf32,
            torch.backends.cudnn.allow_tf32,
        ) = old


# ============================================================
# ЗАПУСК И ИТОГОВАЯ ТАБЛИЦА
# ============================================================

if __name__ == "__main__":
    from IPython.display import display
    import pandas as pd

    current_directory = Path.cwd().resolve()
    project_root = (
        current_directory.parent
        if current_directory.name == "notebooks"
        else current_directory
    )

    resized_reports = run_resized_benchmarks(
        project_root,
        device="cuda:0",  # Выделенная свободная GPU.
    )

    resized_summary = pd.DataFrame([
        {
            "model": name,
            "validation_F1": r["validation"]["validation_f1_mean"],
            "test_F1": r["test"]["test_f1_mean"],
            "test_F1_std": r["test"]["test_f1_std"],
            "test_IoU": r["test"]["test_iou_mean"],
            "test_IoU_std": r["test"]["test_iou_std"],

            **{
                key: r["benchmark"][key]
                for key in (
                    "parameters",
                    "weights_only_mb",
                    "checkpoint_mb",
                    "latency_ms_per_pair",
                    "pairs_per_second",
                    "peak_inference_vram_mb",
                )
            },
        }
        for name, r in resized_reports.items()
    ])

    display(resized_summary.round(6))


light_resized | fc_siam_diff | 256×256 | FP32
Deployment seed: 42; threshold: 0.7
Рабочая папка: artifacts/debug/light_resized_benchmark_eb6602b9
validation seed= 42 | F1=0.754856 | IoU=0.606240
validation seed=123 | F1=0.726672 | IoU=0.570687
validation seed=777 | F1=0.740126 | IoU=0.587461
test       seed= 42 | F1=0.756713 | IoU=0.608639
test       seed=123 | F1=0.721896 | IoU=0.564818
test       seed=777 | F1=0.733585 | IoU=0.579261
Сохранено: artifacts/final_models/light_resized

medium_resized | bit | 256×256 | FP32
Deployment seed: 123; threshold: 0.7
Рабочая папка: artifacts/debug/medium_resized_benchmark_276d5a71
validation seed= 42 | F1=0.709282 | IoU=0.549525
validation seed=123 | F1=0.718927 | IoU=0.561191


RuntimeError: Validation отличается от сохранённой: medium_resized, seed=123. Test не запущен. Смотри artifacts/debug/medium_resized_benchmark_276d5a71

In [6]:
# MEDIUM RESIZED ONLY: validation -> test -> inference benchmark -> export.
# Paste this entire block into a new notebook cell. No training is performed.

import gc
import json
import math
import shutil
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import torch
from satchange import BCEDiceLoss
from satchange.data import create_dataset_dataloader
from satchange.models import count_model_parameters, create_change_model
from satchange.training import evaluate_thresholds
from satchange.utils import load_config


def benchmark_medium_resized(project_root, device="cuda:0"):
    root, device = Path(project_root).resolve(), torch.device(device)
    seeds = (42, 123, 777)
    metric_names = ("loss", "precision", "recall", "f1", "iou", "accuracy")
    search_name = "bit_medium_resized_final_100ep_seeds42_123_777_v1"
    run_root = root / "artifacts/hyperparameter_search" / search_name
    destination = root / "artifacts/final_models/medium_resized"
    source_path = run_root / "results.json"
    if not source_path.is_file():
        source_path = destination / "training_results.json"
    source = json.loads(source_path.read_text(encoding="utf-8-sig"))
    runs = sorted(source["results"], key=lambda r: r["parameters"]["seed"])
    if (source["model_name"] != "bit" or source["search_name"] != search_name
            or [r["parameters"]["seed"] for r in runs] != list(seeds)
            or any(r["status"] != "completed" for r in runs)):
        raise RuntimeError(f"Не совпали модель, final run или seeds: {source_path}")

    architecture = runs[0]["parameters"]["model_parameters"]
    if any(r["parameters"]["image_size"] != 256
           or r["parameters"]["batch_size"] != 16
           or r["parameters"].get("precision", "fp32") != "fp32"
           or r["parameters"]["model_parameters"] != architecture for r in runs):
        raise RuntimeError("Ожидались финальные BIT Resized: 256×256, batch=16, FP32.")
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Выбери notebook kernel с доступом к GPU.")

    # YAML нужен только для пути к данным, не для архитектуры или loss.
    config = load_config(root / "configs/bit_resized.yaml")
    dataset_root = root / config["dataset"]["prepared_directory"]
    winner = max(runs, key=lambda r: r["best_f1"])  # Выбор ДО test.
    winner_seed = winner["parameters"]["seed"]
    work = root / "artifacts/debug" / f"medium_resized_benchmark_{uuid4().hex[:8]}"
    staging = work / "ready"
    if destination.exists():
        shutil.copytree(destination, staging)
    else:
        staging.mkdir(parents=True)

    def save(path, value):
        path.write_text(json.dumps(value, indent=2, ensure_ascii=False,
                                   allow_nan=False) + "\n", encoding="utf-8")

    checkpoint_sources = {}
    for run in runs:
        seed = run["parameters"]["seed"]
        path = destination / f"seed_{seed}_best.pt"
        if not path.is_file():
            path = run_root / "bit" / f"trial_{int(run['trial']):03d}" / "best.pt"
        shutil.copy2(path, staging / f"seed_{seed}_best.pt")
        checkpoint_sources[str(seed)] = str(path.relative_to(root))

    old_flags = (torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic,
                 torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    old_matmul_precision = torch.get_float32_matmul_precision()
    model = loss = before = after = output = None
    reports = {}

    try:
        # Тот же протокол оценки, что в завершённом прогоне Light Resized.
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        protocol = {
            "precision": "fp32", "tf32_matmul": False, "tf32_cudnn": False,
            "cudnn_benchmark": False, "cudnn_deterministic": True,
            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
            "device": str(device), "torch_version": str(torch.__version__),
            "cuda_version": torch.version.cuda,
            "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        }
        gc.collect()
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
            torch.cuda.synchronize(device)
        baseline = torch.cuda.memory_allocated(device) if device.type == "cuda" else 0
        model = create_change_model("bit", architecture).float().to(device)

        def restore(run):
            seed = run["parameters"]["seed"]
            # Только собственные доверенные checkpoints проекта.
            cp = torch.load(staging / f"seed_{seed}_best.pt",
                            map_location="cpu", weights_only=False)
            saved_metrics = cp["current_metrics"]
            if (cp["model_name"] != "bit" or cp["epoch"] != run["best_epoch"]
                    or abs(saved_metrics["best_threshold"] - run["best_threshold"]) > 1e-12
                    or any(abs(saved_metrics[f"validation_{m}"] - run[f"best_{m}"]) > 1e-6
                           for m in ("f1", "iou"))):
                raise RuntimeError(f"Checkpoint не соответствует исходному отчёту: seed={seed}")
            for key in ("seed", "image_size", "model_parameters", "batch_size",
                        "positive_weight", "bce_weight", "dice_weight"):
                if key in cp.get("config", {}) and cp["config"][key] != run["parameters"][key]:
                    raise RuntimeError(f"Checkpoint: не совпал {key}, seed={seed}")
            if cp.get("precision", "fp32") != "fp32":
                raise RuntimeError(f"Checkpoint: неожиданная precision, seed={seed}")
            model.load_state_dict(cp["model_state_dict"], strict=True)
            model.eval()

        print("Medium Resized | BIT | 256×256 | FP32 | TF32 выключен")
        print(f"Deployment seed={winner_seed}; threshold={winner['best_threshold']}")
        print("Light пропущена. Рабочая папка:", work)

        for split, filename in (("validation", "validation_results.json"),
                                ("test", "final_test_results.json")):
            dataset, loader = create_dataset_dataloader(
                dataset_root=dataset_root, split="val" if split == "validation" else "test",
                image_size=256, batch_size=16, num_workers=0, transform=None,
                shuffle=False, pin_memory=device.type == "cuda",
            )
            rows = []
            for run in runs:
                p = run["parameters"]
                restore(run)
                loss = BCEDiceLoss(bce_weight=p["bce_weight"], dice_weight=p["dice_weight"],
                                   pos_weight=p["positive_weight"]).to(device)
                with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=False):
                    result = evaluate_thresholds(
                        model=model, data_loader=loader, loss_function=loss, device=device,
                        thresholds=[float(run["best_threshold"])], precision="fp32",
                    )
                metrics = {"loss": result["loss"], **result["threshold_results"][0]}
                if not all(math.isfinite(float(metrics[k])) for k in metric_names):
                    raise RuntimeError(f"NaN/Inf в метриках: {split}, seed={p['seed']}")
                row = {
                    "seed": p["seed"], "trial": f"trial_{int(run['trial']):03d}",
                    "best_validation_epoch": run["best_epoch"],
                    "validation_threshold": run["best_threshold"],
                    "original_validation_f1": run["best_f1"],
                    "original_validation_iou": run["best_iou"],
                    **{f"{split}_{k}": float(metrics[k]) for k in metric_names},
                    **{f"{split}_{k}": int(metrics[k]) for k in ("tp", "fp", "fn", "tn")},
                }
                if split == "validation":
                    for m in ("f1", "iou"):
                        row[f"validation_{m}_delta"] = float(metrics[m]) - run[f"best_{m}"]
                    # Другая вычислительная настройка обучения: сравнение информативное.
                    # Старые метрики не заменяются и не подставляются вместо новых.
                    row["historical_difference_warning"] = any(
                        abs(row[f"validation_{m}_delta"]) > 1e-4 for m in ("f1", "iou")
                    )
                    if row["historical_difference_warning"]:
                        print(f"  Предупреждение seed={p['seed']}: "
                              f"ΔF1={row['validation_f1_delta']:+.8f}, "
                              f"ΔIoU={row['validation_iou_delta']:+.8f}; продолжаем.")
                else:
                    val = next(v for v in reports["validation"]["results"] if v["seed"] == p["seed"])
                    row.update(validation_f1=val["validation_f1"], validation_iou=val["validation_iou"])
                rows.append(row)
                save(work / f"{split}_progress.json", rows)
                print(f"{split:10s} seed={p['seed']:3d} | "
                      f"F1={metrics['f1']:.6f} | IoU={metrics['iou']:.6f}")

            report = {
                "model": "bit", "variant": "medium_resized", "split": split,
                "image_size": 256, "number_of_pairs": len(dataset), "seeds": list(seeds),
                "evaluation_batch_size": 16, "num_workers": 0, "precision": "fp32",
                "std_type": "sample", "std_ddof": 1, "protocol": protocol,
                "threshold_selection": "saved validation thresholds; no new threshold search",
                "historical_validation_comparison": "informational; not an equality gate",
                "historical_backend_note": "training backend settings differed, as reported by user; exact values unknown",
                "results": rows,
            }
            for m in metric_names:
                key = f"{split}_{m}"
                report[f"{key}_mean"] = statistics.mean(r[key] for r in rows)
                report[f"{key}_std"] = statistics.stdev(r[key] for r in rows)
            reports[split] = report
            save(staging / filename, report)
        del loader, dataset
        loss = None
        restore(winner)
        torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()},
                   staging / "deployment_weights.pt")
        gc.collect()
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()

        # Inference benchmark: batch=1, без I/O, resize и thresholding.
        before = torch.zeros(1, 3, 256, 256, device=device)
        after = torch.zeros_like(before)
        times = []
        with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=False):
            for _ in range(20):
                output = model(before, after)
                del output
            if device.type == "cuda":
                torch.cuda.synchronize(device)
                torch.cuda.reset_peak_memory_stats(device)
            for _ in range(100):
                if device.type == "cuda":
                    torch.cuda.synchronize(device)
                started = time.perf_counter()
                output = model(before, after)
                if device.type == "cuda":
                    torch.cuda.synchronize(device)
                times.append((time.perf_counter() - started) * 1000)
                del output
        latency = statistics.mean(times)
        counts = count_model_parameters(model)
        peak = torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None
        benchmark = {
            **protocol, "parameters": counts["total_parameters"],
            "trainable_parameters": counts["trainable_parameters"],
            "weights_only_mb": (staging / "deployment_weights.pt").stat().st_size / 2**20,
            "checkpoint_mb": (staging / f"seed_{winner_seed}_best.pt").stat().st_size / 2**20,
            "checkpoint_sizes_mib": {str(s): (staging / f"seed_{s}_best.pt").stat().st_size / 2**20 for s in seeds},
            "latency_ms_per_pair": latency, "latency_median_ms": statistics.median(times),
            "pairs_per_second": 1000 / latency, "latency_samples_ms": times,
            "peak_inference_vram_mb": (peak - baseline) / 2**20 if peak is not None else None,
            "notebook_baseline_vram_mb": baseline / 2**20 if peak is not None else None,
            "process_peak_allocated_vram_mb": peak / 2**20 if peak is not None else None,
            "process_peak_reserved_vram_mb": (torch.cuda.max_memory_reserved(device) / 2**20
                                                if device.type == "cuda" else None),
            "protocol": {**protocol, "tf32": False, "batch_size": 1, "warmup": 20,
                         "repetitions": 100, "input_shape_per_image": [1, 3, 256, 256],
                         "images_per_pair": 2, "timing": "synchronized forward only",
                         "memory_unit": "MiB; pre-existing notebook allocations subtracted from inference peak"},
        }
        reports["benchmark"] = benchmark
        save(staging / "benchmark.json", benchmark)
        save(staging / "training_results.json", source)
        val_by_seed = {r["seed"]: r for r in reports["validation"]["results"]}
        metadata = {
            "model_level": "medium", "variant": "resized_256", "model_name": "bit",
            "image_size": 256, "model_parameters": architecture,
            "deployment_seed": winner_seed, "deployment_best_epoch": winner["best_epoch"],
            "deployment_threshold": winner["best_threshold"],
            "deployment_selection": "highest ORIGINAL validation F1; never test F1",
            "deployment_validation_f1": val_by_seed[winner_seed]["validation_f1"],
            "deployment_validation_iou": val_by_seed[winner_seed]["validation_iou"],
            "deployment_original_validation_f1": winner["best_f1"],
            "deployment_original_validation_iou": winner["best_iou"],
            "evaluation_protocol": protocol, "evaluation_complete": True,
            "source_results": str(source_path.relative_to(root)), "dataset_root": str(dataset_root),
            "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
            "seeds": {str(r["parameters"]["seed"]): {
                **val_by_seed[r["parameters"]["seed"]], "best_epoch": r["best_epoch"],
                "parameters": r["parameters"],
                "source_checkpoint": checkpoint_sources[str(r["parameters"]["seed"])],
            } for r in runs},
        }
        save(staging / "metadata.json", metadata)

        # Финальная папка меняется только после успешного завершения всех этапов.
        backup = work / "previous"
        destination.parent.mkdir(parents=True, exist_ok=True)
        if destination.exists():
            destination.rename(backup)
        try:
            staging.rename(destination)
        except BaseException:
            if backup.exists():
                backup.rename(destination)
            raise
        print("\nСохранено:", destination)
        print(f"Test F1:  {reports['test']['test_f1_mean']:.6f} ± {reports['test']['test_f1_std']:.6f}")
        print(f"Test IoU: {reports['test']['test_iou_mean']:.6f} ± {reports['test']['test_iou_std']:.6f}")
        print(f"Latency: {latency:.3f} ms/pair | {1000 / latency:.3f} pairs/s")
        return reports
    finally:
        model = loss = before = after = output = None
        gc.collect()
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
        (torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic,
         torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32) = old_flags
        torch.set_float32_matmul_precision(old_matmul_precision)


# Запусти только эту ячейку, не весь notebook с HPO.
if __name__ == "__main__":
    current = Path.cwd().resolve()
    project_root = current.parent if current.name == "notebooks" else current
    medium_resized_reports = benchmark_medium_resized(project_root, device="cuda:0")


Medium Resized | BIT | 256×256 | FP32 | TF32 выключен
Deployment seed=123; threshold=0.7
Light пропущена. Рабочая папка: artifacts/debug/medium_resized_benchmark_e683e1d5
validation seed= 42 | F1=0.709282 | IoU=0.549525
  Предупреждение seed=123: ΔF1=+0.00010186, ΔIoU=+0.00012412; продолжаем.
validation seed=123 | F1=0.718927 | IoU=0.561191
validation seed=777 | F1=0.704820 | IoU=0.544187
test       seed= 42 | F1=0.684597 | IoU=0.520446
test       seed=123 | F1=0.706518 | IoU=0.546214
test       seed=777 | F1=0.661565 | IoU=0.494283

Сохранено: artifacts/final_models/medium_resized
Test F1:  0.684227 ± 0.022479
Test IoU: 0.520314 ± 0.025966
Latency: 16.762 ms/pair | 59.658 pairs/s


In [9]:
# LIGHT FULL: validation -> test -> inference benchmark -> final_models.
# Запустить целиком в одной новой ячейке. Обучения здесь нет.

import gc
import hashlib
import json
import math
import shutil
import statistics
import time
from datetime import datetime, timezone
from pathlib import Path
from uuid import uuid4

import torch
from satchange import BCEDiceLoss
from satchange.data import create_dataset_dataloader
from satchange.models import count_model_parameters, create_change_model
from satchange.training import evaluate_thresholds
from satchange.utils import load_config


def _lf_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False,
                               allow_nan=False) + "\n", encoding="utf-8")


def _lf_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


@torch.inference_mode()
def _lf_forward_benchmark(model, device, baseline):
    # Только forward. Входы уже на устройстве, batch=1.
    before = torch.zeros(1, 3, 1024, 1024, device=device, dtype=torch.float32)
    after = torch.zeros_like(before)
    model.eval()
    with torch.autocast(device_type=device.type, enabled=False):
        for _ in range(20):
            output = model(before, after)
            del output
        if device.type == "cuda":
            torch.cuda.synchronize(device)
            torch.cuda.reset_peak_memory_stats(device)
        samples = []
        for _ in range(100):
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            started = time.perf_counter()
            output = model(before, after)
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            samples.append((time.perf_counter() - started) * 1000)
            del output
    peak = torch.cuda.max_memory_allocated(device) if device.type == "cuda" else None
    latency = statistics.mean(samples)
    return {
        "latency_ms_per_pair": latency,
        "latency_median_ms": statistics.median(samples),
        "pairs_per_second": 1000 / latency,
        "latency_samples_ms": samples,
        "peak_inference_vram_mb": (peak - baseline) / 2**20 if peak is not None else None,
        "notebook_baseline_vram_mb": baseline / 2**20 if peak is not None else None,
        "process_peak_allocated_vram_mb": peak / 2**20 if peak is not None else None,
        "process_peak_reserved_vram_mb": (torch.cuda.max_memory_reserved(device) / 2**20
                                          if device.type == "cuda" else None),
    }


def benchmark_light_full(project_root, device="cuda:0"):
    root, device = Path(project_root).resolve(), torch.device(device)
    destination = root / "artifacts/final_models/light_full"
    search_name = "fc_siam_diff_full_candidate_c_final_multiseed_deterministic"
    run_root = root / "artifacts/training_runs" / search_name
    source_path = run_root / "results.json"
    if not source_path.is_file():
        source_path = destination / "training_results.json"
    if not source_path.is_file():
        raise FileNotFoundError(f"Нужен исходный отчёт: {run_root / 'results.json'}")

    source = json.loads(source_path.read_text(encoding="utf-8-sig"))
    runs = sorted(source["results"], key=lambda r: r["parameters"]["seed"])
    seeds = [r["parameters"]["seed"] for r in runs]
    if (source["model_name"] != "fc_siam_diff" or source["search_name"] != search_name
            or seeds != [123, 777, 2024]
            or any(r["status"] != "completed" for r in runs)):
        raise RuntimeError(f"Это не финальный Light Full candidate C: {source_path}")
    architecture = runs[0]["parameters"]["model_parameters"]
    if any(r["parameters"]["image_size"] is not None
           or r["parameters"]["model_parameters"] != architecture for r in runs):
        raise RuntimeError("Ожидались одинаковая архитектура и image_size=None (без resize).")
    if device.type == "cuda" and not torch.cuda.is_available():
        raise RuntimeError("CUDA недоступна. Выбери kernel с доступом к GPU.")

    # Из YAML только путь к данным: в старом YAML другая ширина модели!
    config = load_config(root / "configs/fc_siam_diff_full.yaml")
    dataset_root = root / config["dataset"]["prepared_directory"]
    winner = max(runs, key=lambda r: r["best_f1"])  # До test.
    winner_seed = winner["parameters"]["seed"]
    work = root / "artifacts/debug" / f"light_full_benchmark_{uuid4().hex[:8]}"
    staging = work / "ready"
    if destination.exists():
        shutil.copytree(destination, staging)
    else:
        staging.mkdir(parents=True)
    old_metadata_path = staging / "metadata.json"
    metadata = json.loads(old_metadata_path.read_text(encoding="utf-8-sig")) if old_metadata_path.is_file() else {}
    old_test_path = staging / "final_test_results.json"
    old_test = json.loads(old_test_path.read_text(encoding="utf-8-sig")) if old_test_path.is_file() else {}
    old_test_rows = {int(r["seed"]): r for r in old_test.get("results", [])}
    checkpoint_sources, hashes = {}, {}
    for run in runs:
        seed = run["parameters"]["seed"]
        path = destination / f"seed_{seed}_best.pt"
        if not path.is_file():
            path = run_root / "fc_siam_diff" / f"trial_{int(run['trial']):03d}" / "best.pt"
        target = staging / f"seed_{seed}_best.pt"
        if not target.is_file():
            shutil.copy2(path, target)
        checkpoint_sources[str(seed)] = str(path.relative_to(root))
        hashes[str(seed)] = _lf_sha256(target)

    old_flags = (torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic,
                 torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32)
    old_precision = torch.get_float32_matmul_precision()
    model = loss = None
    metrics_names = ("loss", "precision", "recall", "f1", "iou", "accuracy")
    reports = {}
    try:
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        protocol = {
            "precision": "fp32", "tf32_matmul": False, "tf32_cudnn": False,
            "cudnn_benchmark": False, "cudnn_deterministic": True,
            "deterministic_algorithms": torch.are_deterministic_algorithms_enabled(),
            "device": str(device), "torch_version": str(torch.__version__),
            "cuda_version": torch.version.cuda, "cudnn_version": torch.backends.cudnn.version(),
            "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        }
        gc.collect()
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
            torch.cuda.synchronize(device)
        baseline = torch.cuda.memory_allocated(device) if device.type == "cuda" else 0
        model = create_change_model("fc_siam_diff", architecture).float().to(device)

        def restore(run):
            seed = run["parameters"]["seed"]
            # Только собственные доверенные checkpoints проекта.
            cp = torch.load(staging / f"seed_{seed}_best.pt", map_location="cpu", weights_only=False)
            saved = cp["current_metrics"]
            if (cp["model_name"] != "fc_siam_diff" or cp["epoch"] != run["best_epoch"]
                    or not math.isclose(saved["best_threshold"], run["best_threshold"], rel_tol=0, abs_tol=1e-12)
                    or any(not math.isclose(saved[f"validation_{m}"], run[f"best_{m}"], rel_tol=0, abs_tol=1e-6)
                           for m in ("f1", "iou"))):
                raise RuntimeError(f"Checkpoint не соответствует исходному отчёту: seed={seed}")
            for key in ("seed", "image_size", "model_parameters", "batch_size",
                        "positive_weight", "bce_weight", "dice_weight"):
                if key in cp.get("config", {}) and cp["config"][key] != run["parameters"][key]:
                    raise RuntimeError(f"Checkpoint: не совпал {key}, seed={seed}")
            model.load_state_dict(cp["model_state_dict"], strict=True)
            model.eval()

        print("Light Full | FC-Siam-Diff | исходные 1024×1024 | FP32, TF32 выключен")
        print("Исходный отчёт:", source_path)
        print("Архитектура:", architecture)
        print(f"Seeds: {seeds}; deployment={winner_seed}; threshold={winner['best_threshold']}")
        print("Рабочая папка:", work)
        for split, filename in (("validation", "validation_results.json"), ("test", "final_test_results.json")):
            rows = []
            for run in runs:
                p, seed = run["parameters"], run["parameters"]["seed"]
                restore(run)
                dataset, loader = create_dataset_dataloader(
                    dataset_root=dataset_root, split="val" if split == "validation" else "test",
                    image_size=None, batch_size=p["batch_size"], num_workers=0,
                    transform=None, shuffle=False, pin_memory=device.type == "cuda",
                )
                loss = BCEDiceLoss(bce_weight=p["bce_weight"], dice_weight=p["dice_weight"],
                                  pos_weight=p["positive_weight"]).to(device)
                with torch.inference_mode(), torch.autocast(device_type=device.type, enabled=False):
                    result = evaluate_thresholds(model=model, data_loader=loader, loss_function=loss,
                                                 device=device, thresholds=[float(run["best_threshold"])], precision="fp32")
                measured = {"loss": result["loss"], **result["threshold_results"][0]}
                if not all(math.isfinite(float(measured[m])) for m in metrics_names):
                    raise RuntimeError(f"NaN/Inf: {split}, seed={seed}")
                row = {
                    "seed": seed, "trial": f"trial_{int(run['trial']):03d}",
                    "best_validation_epoch": run["best_epoch"], "validation_threshold": run["best_threshold"],
                    "evaluation_batch_size": p["batch_size"], "source_checkpoint": checkpoint_sources[str(seed)],
                    "original_validation_f1": run["best_f1"], "original_validation_iou": run["best_iou"],
                    **{f"{split}_{m}": float(measured[m]) for m in metrics_names},
                    **{f"{split}_{m}": int(measured[m]) for m in ("tp", "fp", "fn", "tn")},
                }
                if split == "validation":
                    for m in ("f1", "iou"):
                        row[f"validation_{m}_delta"] = float(measured[m]) - run[f"best_{m}"]
                    row["historical_difference_warning"] = any(abs(row[f"validation_{m}_delta"]) > 1e-4 for m in ("f1", "iou"))
                    if row["historical_difference_warning"]:
                        print(f"  Изменение относительно истории: ΔF1={row['validation_f1_delta']:+.8f}, "
                              f"ΔIoU={row['validation_iou_delta']:+.8f}. Сохраняем, не останавливаемся.")
                else:
                    val = next(r for r in reports["validation"]["results"] if r["seed"] == seed)
                    for m in ("f1", "iou"):
                        row[f"validation_{m}"] = val[f"validation_{m}"]
                        previous = old_test_rows.get(seed, {}).get(f"test_{m}")
                        if previous is not None:
                            row[f"previous_test_{m}"] = previous
                            row[f"test_{m}_delta"] = float(measured[m]) - previous
                rows.append(row)
                _lf_json(work / f"{split}_progress.json", rows)
                print(f"{split:10s} seed={seed:4d} | F1={measured['f1']:.6f} | IoU={measured['iou']:.6f}")
            report = {
                "model": "fc_siam_diff", "variant": "light_full", "candidate": "C", "split": split,
                "image_size": None, "native_resolution": [1024, 1024], "seeds": seeds,
                "number_of_pairs": len(dataset), "num_workers": 0, "precision": "fp32",
                "evaluation_batch_size": runs[0]["parameters"]["batch_size"],
                "std_type": "sample", "std_ddof": 1, "protocol": protocol,
                "threshold_selection": "original validation thresholds; no new threshold search",
                "historical_comparison": "informational; not proof of identical execution conditions",
                "results": rows,
            }
            for m in metrics_names:
                key = f"{split}_{m}"
                report[f"{key}_mean"] = statistics.mean(r[key] for r in rows)
                report[f"{key}_std"] = statistics.stdev(r[key] for r in rows)
            reports[split] = report
            _lf_json(staging / filename, report)

        del loader, dataset
        loss = None
        restore(winner)
        # Содержимое deployment должно соответствовать ранее выбранному seed.
        weights = staging / "deployment_weights.pt"
        if weights.is_file():
            existing = torch.load(weights, map_location="cpu", weights_only=True)
            current = model.state_dict()
            if set(existing) != set(current) or any(not torch.equal(existing[k], current[k].detach().cpu()) for k in current):
                raise RuntimeError("deployment_weights.pt не совпадает с выбранным checkpoint; файлы не заменены.")
            del existing, current
        else:
            torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, weights)
        gc.collect()
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()
        benchmark = _lf_forward_benchmark(model, device, baseline)
        counts = count_model_parameters(model)
        benchmark.update({
            **protocol, "parameters": counts["total_parameters"], "trainable_parameters": counts["trainable_parameters"],
            "weights_only_mb": weights.stat().st_size / 2**20,
            "checkpoint_mb": (staging / f"seed_{winner_seed}_best.pt").stat().st_size / 2**20,
            "checkpoint_sizes_mib": {str(s): (staging / f"seed_{s}_best.pt").stat().st_size / 2**20 for s in seeds},
            "protocol": {**protocol, "batch_size": 1, "warmup": 20, "repetitions": 100,
                         "input_shape_per_image": [1, 3, 1024, 1024], "images_per_pair": 2,
                         "timing": "synchronized forward only; no I/O, resize or thresholding",
                         "throughput": "sequential batch=1 pairs/s, not maximum batched throughput",
                         "memory_unit": "MiB; notebook baseline subtracted from inference peak"},
        })
        reports["benchmark"] = benchmark
        _lf_json(staging / "benchmark.json", benchmark)
        _lf_json(staging / "training_results.json", source)  # Оригинальные результаты, без подмены.
        deployment_val = next(r for r in reports["validation"]["results"] if r["seed"] == winner_seed)
        metadata.update({
            "model_level": "light", "variant": "full_resolution", "model_name": "fc_siam_diff",
            "image_size": None, "native_resolution": [1024, 1024], "model_parameters": architecture,
            "deployment_seed": winner_seed, "deployment_threshold": winner["best_threshold"],
            "deployment_best_epoch": winner["best_epoch"],
            "deployment_validation_f1": deployment_val["validation_f1"],
            "deployment_validation_iou": deployment_val["validation_iou"],
            "deployment_original_validation_f1": winner["best_f1"],
            "deployment_original_validation_iou": winner["best_iou"],
            "evaluation_function": "satchange.training.evaluate_thresholds",
            "deployment_selection": "highest ORIGINAL validation F1; not test F1",
            "source_results": str(source_path.relative_to(root)), "checkpoint_sha256": hashes,
            "dataset_root": str(dataset_root), "evaluation_complete": True, "evaluation_protocol": protocol,
            "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
            "seeds": {str(r["seed"]): {**r, "best_epoch": r["best_validation_epoch"],
                       "parameters": next(x["parameters"] for x in runs if x["parameters"]["seed"] == r["seed"])}
                      for r in reports["validation"]["results"]},
        })
        _lf_json(staging / "metadata.json", metadata)
        destination.parent.mkdir(parents=True, exist_ok=True)
        backup = work / "previous"
        if destination.exists():
            destination.rename(backup)
        try:
            staging.rename(destination)
        except BaseException:
            if backup.exists():
                backup.rename(destination)
            raise
        print("\nСохранено:", destination)
        print("Прежняя папка:", backup if backup.exists() else "отсутствовала")
        print(f"Test F1:  {reports['test']['test_f1_mean']:.6f} ± {reports['test']['test_f1_std']:.6f}")
        print(f"Test IoU: {reports['test']['test_iou_mean']:.6f} ± {reports['test']['test_iou_std']:.6f}")
        print(f"Latency: {benchmark['latency_ms_per_pair']:.3f} ms/pair | {benchmark['pairs_per_second']:.3f} pairs/s")
        print(f"Параметров: {counts['total_parameters']:,}; веса: {benchmark['weights_only_mb']:.2f} MiB")
        print("Peak inference VRAM, MiB:", benchmark["peak_inference_vram_mb"])
        return reports
    finally:
        model = loss = None
        gc.collect()
        # Сначала возвращаем настройки, даже если последующая очистка CUDA даст ошибку.
        torch.set_float32_matmul_precision(old_precision)
        (torch.backends.cudnn.benchmark, torch.backends.cudnn.deterministic,
         torch.backends.cuda.matmul.allow_tf32, torch.backends.cudnn.allow_tf32) = old_flags
        if device.type == "cuda":
            with torch.cuda.device(device):
                torch.cuda.empty_cache()


# Запусти только эту ячейку. Остальные модели не оцениваются.
if __name__ == "__main__":
    current = Path.cwd().resolve()
    project_root = current.parent if current.name == "notebooks" else current
    light_full_reports = benchmark_light_full(project_root, device="cuda:0")


Light Full | FC-Siam-Diff | исходные 1024×1024 | FP32, TF32 выключен
Исходный отчёт: artifacts/training_runs/fc_siam_diff_full_candidate_c_final_multiseed_deterministic/results.json
Архитектура: {'in_channels': 3, 'out_channels': 1, 'base_channels': 32}
Seeds: [123, 777, 2024]; deployment=2024; threshold=0.8
Рабочая папка: artifacts/debug/light_full_benchmark_f2a4b13f
validation seed= 123 | F1=0.823480 | IoU=0.699929
validation seed= 777 | F1=0.769524 | IoU=0.625387
validation seed=2024 | F1=0.850922 | IoU=0.740526
test       seed= 123 | F1=0.810196 | IoU=0.680949
test       seed= 777 | F1=0.764804 | IoU=0.619176
test       seed=2024 | F1=0.841149 | IoU=0.725848

Сохранено: artifacts/final_models/light_full
Прежняя папка: artifacts/debug/light_full_benchmark_f2a4b13f/previous
Test F1:  0.805383 ± 0.038399
Test IoU: 0.675324 ± 0.053558
Latency: 128.565 ms/pair | 7.778 pairs/s
Параметров: 7,760,097; веса: 29.62 MiB
Peak inference VRAM, MiB: 1309.70458984375
